# 🦄 VenturePulse: Global Startup Valuation & Capital Efficiency
### CMSE 830 - Midterm Project Analysis & Empirical Modeling

**Research Question:** *To what extent can private tech startup valuations be quantitatively modeled using capital raised, sector, geography, age, and investor network—and who are the true capital-efficient outliers?*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Configure plots
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")

# Load processed dataset
df = pd.read_csv("../data/processed/unicorns_with_predictions.csv")
print(f"Loaded {len(df)} unicorns across {df['country'].nunique()} countries.")
df.head(3)

## 1. Exploratory Data Analysis: Valuation Distributions & Power Laws
Private tech valuations follow an extreme **Pareto / Power Law distribution**. Below we contrast the raw skewed distribution with the log-transformed distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["valuation_b"], bins=30, kde=True, ax=axes[0], color="#3b82f6")
axes[0].set_title("Raw Valuation Distribution ($B) - Extreme Positive Skew")
axes[0].set_xlabel("Valuation ($B)")

sns.histplot(np.log1p(df["valuation_m"]), bins=30, kde=True, ax=axes[1], color="#10b981")
axes[1].set_title("Log-Transformed Valuation - Approximate Normality")
axes[1].set_xlabel("log(Valuation in $M)")

plt.tight_layout()
plt.show()

## 2. Sector Valuation Multiples & Capital Raised
Examining how total equity capital raised scales into post-money valuation across different tech sectors.

In [ ]:
plt.figure(figsize=(12, 6))
top_sectors = df["clean_industry"].value_counts().head(8).index
subset = df[df["clean_industry"].isin(top_sectors)]

sns.boxplot(data=subset, x="clean_industry", y="capital_efficiency_ratio", showfliers=False, palette="viridis")
plt.title("Capital Efficiency Ratio (Valuation / Total Raised) by Sector")
plt.xlabel("Sector")
plt.ylabel("Valuation / Funding Multiple (x)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 3. Machine Learning Model Benchmark & Diagnostics
Reviewing the trained Random Forest and Ridge regression pipelines.

In [ ]:
with open("../models/valuation_pipeline.pkl", "rb") as f:
    bundle = pickle.load(f)

print(f"Selected Model: {bundle['model_name']}")
print(f"5-Fold CV R²: {bundle['cv_r2']:.4f}")
print(f"Test Set R²:   {bundle['test_r2']:.4f}")
print(f"Test Set MAE:  ${bundle['test_mae_m']:,.2f} Million\n")

print("Top 8 Positive Valuation Elasticities (Ridge):")
print(bundle["coefficients"].head(8).to_string(index=False))

## 4. Capital Efficiency Residual Analysis
Examining companies that deviate substantially from expected market valuations.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="total_raised_m_imputed",
    y="valuation_b",
    hue="efficiency_tier",
    palette={"Capital-Efficient Alpha (Undervalued / High Yield)": "#10b981", "Fair Market Valuation": "#3b82f6", "Capital Guzzler (Over-Funded Relative to Value)": "#ef4444"},
    alpha=0.7
)
plt.xscale("log")
plt.yscale("log")
plt.title("Capital Efficiency Matrix: Funding Raised vs Actual Valuation (Log Scale)")
plt.xlabel("Total Equity Capital Raised ($M, Log Scale)")
plt.ylabel("Valuation ($B, Log Scale)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()